<a href="https://colab.research.google.com/github/angie05huang/angie05huang/blob/main/SpeedDatingModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install liac-arff pandas

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving speeddating.arff to speeddating (11).arff


In [ ]:
import arff
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

with open("speeddating.arff", "r") as f:
    arff_data = arff.load(f)

cols = [a[0] for a in arff_data['attributes']]
df = pd.DataFrame(arff_data['data'], columns=cols)

df.to_csv("speeddating.csv", index=False)

df.head()

,has_null,wave,gender,age,age_o,d_age,d_d_age,race,race_o,samerace,...,d_expected_num_interested_in_me,d_expected_num_matches,like,guess_prob_liked,d_like,d_guess_prob_liked,met,decision,decision_o,match
0,0,1.0,female,21.0,27.0,6.0,[4-6],Asian/Pacific Islander/Asian-American,European/Caucasian-American,0,...,[0-3],[3-5],7.0,6.0,[6-8],[5-6],0.0,1,0,0
1,0,1.0,female,21.0,22.0,1.0,[0-1],Asian/Pacific Islander/Asian-American,European/Caucasian-American,0,...,[0-3],[3-5],7.0,5.0,[6-8],[5-6],1.0,1,0,0
2,1,1.0,female,21.0,22.0,1.0,[0-1],Asian/Pacific Islander/Asian-American,Asian/Pacific Islander/Asian-American,1,...,[0-3],[3-5],7.0,NaN,[6-8],[0-4],1.0,1,1,1
3,0,1.0,female,21.0,23.0,2.0,[2-3],Asian/Pacific Islander/Asian-American,European/Caucasian-American,0,...,[0-3],[3-5],7.0,6.0,[6-8],[5-6],0.0,1,1,1
4,0,1.0,female,21.0,24.0,3.0,[2-3],Asian/Pacific Islander/Asian-American,Latino/Hispanic American,0,...,[0-3],[3-5],6.0,6.0,[6-8],[5-6],0.0,1,1,1


In [ ]:
df = df[df["met"] == 0]
df['match'] = df['match'].astype(int)

personality_cols = [
    'attractive_o', 'sinsere_o', 'intelligence_o', 'funny_o', 'ambitous_o',
    'shared_interests_o',
    'attractive', 'sincere', 'intelligence', 'funny', 'ambition'
]

numeric_df = df.select_dtypes(include='number')
non_personality_df = numeric_df.drop(columns=personality_cols, errors='ignore'
corr_with_match = non_personality_df.corr()[['match']].sort_values('match')
plt.figure(figsize=(4, 12))
sns.heatmap(corr_with_match, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation of Numeric Non-Personality Variables with mat')


In [ ]:
df = df[df["gender"] == "male"]
df["match"] = df["match"].astype(int)
df = df[df["met"] == 0]
y = df["match"]
X = df[['attractive_o', 'sinsere_o', 'intelligence_o', 'funny_o', 'ambitous_o',
    'shared_interests_o','shared_interests_partner']]

In [ ]:
def range_to_mid(x):
   if isinstance(x, str) and "[" in x:
       a, b = x.strip("[]").split("-")
       return (float(a) + float(b)) / 2
   return float(x)


range_columns = [
   'd_importance_same_race', 'd_importance_same_religion', 'd_pref_o_attractive',
   'd_pref_o_sincere', 'd_pref_o_intelligence', 'd_pref_o_funny',
   'd_pref_o_ambitious', 'd_pref_o_shared_interests', 'd_attractive_o',
   'd_sinsere_o', 'd_intelligence_o', 'd_funny_o', 'd_ambitous_o',
   'd_shared_interests_o', 'd_attractive_important', 'd_sincere_important',
   'd_intellicence_important', 'd_funny_important', 'd_ambtition_important',
   'd_shared_interests_important', 'd_attractive', 'd_sincere', 'd_intelligence',
   'd_funny', 'd_ambition', 'd_attractive_partner', 'd_sincere_partner',
   'd_intelligence_partner', 'd_funny_partner', 'd_ambition_partner',
   'd_shared_interests_partner', 'd_sports', 'd_tvsports', 'd_exercise',
   'd_dining', 'd_museums', 'd_art', 'd_hiking', 'd_gaming', 'd_clubbing',
   'd_reading', 'd_tv', 'd_theater', 'd_movies', 'd_concerts', 'd_music',
   'd_shopping', 'd_yoga', 'd_expected_happy_with_sd_people',
   'd_expected_num_interested_in_me', 'd_expected_num_matches',
   'd_like', 'd_guess_prob_liked'
]
for col in range_columns:
   if col in df.columns:
       df[col] = df[col].apply(range_to_mid)
for col in range_columns:
   if col in df.columns:
       df[col] = pd.to_numeric(df[col], errors="coerce")


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 0.8]
}


grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Fit
grid_rf.fit(X_train, y_train)

# Results
print("🔹 Best hyperparameters:", grid_rf.best_params_)

Fitting 5 folds for each of 405 candidates, totalling 2025 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
675 fits failed out of a total of 2025.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
68 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py",

🔹 Best hyperparameters: {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 100}


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nTop 20 Features:")
for i in indices[:20]:
    print(f"{X.columns[i]}: {importances[i]:.4f}")